# REM_Turku dream-affect decoding: baselines, nulls, harness

Handoff for **Paul Barbaste**. Runs top to bottom on Colab. Four rules (details: README):
shuffle labels WITHIN subject; null seeds start at 1; a floor-limited p is a floor, not
a measurement; epoch-level and awakening-level scores differ, every row is labelled.


## 0. Configuration

Nothing to configure. REM_Turku downloads and rebuilds itself in section 1 (public
figshare deposit, md5-checked). body_action needs only the two Drive share links pasted
in section 7. `DATA_DIR` below is internal; leave it alone.


In [1]:
import os, sys, subprocess, hashlib, json, zipfile, io, csv
from pathlib import Path

# When opened straight from GitHub, Colab loads ONLY this notebook; fetch the repo so
# prepare_remturku.py and the harness assets exist next to it.
if not Path("prepare_remturku.py").exists():
    if not Path("readream-riemann-baseline").exists():
        subprocess.run(["git", "clone", "-q",
                        "https://github.com/hollanderski/readream-riemann-baseline"],
                       check=True)
    os.chdir("readream-riemann-baseline/notebooks")
    print("working dir:", os.getcwd())

DATA_DIR = Path(os.environ.get("REMTURKU_DIR", "./remturku_data"))
DATA_DIR.mkdir(parents=True, exist_ok=True)
SEEDS_START_AT = 1          # seed 0 means no shuffle; never let it into a null loop


working dir: /content/readream-riemann-baseline/notebooks


In [ ]:
%pip -q install pyriemann==0.12 braindecode mne scikit-learn scipy
# TSMNet arm only; comment out if you are not running it:
%pip -q install git+https://github.com/rkobler/TSMNet geoopt


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 153.6/153.6 kB 2.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 679.2/679.2 kB 12.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.5/7.5 MB 68.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 79.1/79.1 kB 5.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 119.9/119.9 kB 9.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 189.8/189.8 kB 15.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 271.6/271.6 kB 18.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 163.9/163.9 kB 12.0 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
ipython 7.34.0 requires jedi>=0.16, which is not installed.
moviepy 1.0.3 requires decorator<5.0,>=4.0.2, but you have decorator 5.3.1 which is incompatible.


## 1. Data

No shipped archive: the cell downloads the public deposit (figshare
10.6084/m9.figshare.23274596.v2, CC-BY, 363 MB, md5-checked) and rebuilds the epochs
with `prepare_remturku.py` (~15-20 min, cached).


In [ ]:
API = "https://api.figshare.com/v2/articles/23274596/versions/2"
zip_path = DATA_DIR / "REM_Turku.zip"
npz_path = DATA_DIR / "remturku_epochs.npz"

def ensure_data():
    """Idempotent: download (md5-checked) + preprocess only if missing. Called
    automatically by load(), so cells work in any order."""
    if npz_path.exists():
        return
    if not zip_path.exists():
        import urllib.request
        meta = json.load(urllib.request.urlopen(API))
        rec = next(f for f in meta["files"] if f["name"] == "REM_Turku.zip")
        print(f"downloading {rec['name']} ({rec['size']/1e6:.0f} MB)...")
        subprocess.run(["curl", "-L", "-o", str(zip_path), rec["download_url"]],
                       check=True)
        assert zip_path.stat().st_size == rec["size"], "size mismatch, re-run this cell"
        md5 = hashlib.md5(zip_path.read_bytes()).hexdigest()
        assert md5 == rec["supplied_md5"], f"md5 mismatch: {md5}"
    print("preprocessing (~15 min, once)...")
    subprocess.run([sys.executable, "prepare_remturku.py", "--zip", str(zip_path),
                    "--out", str(npz_path)], check=True)

ensure_data()
print("epochs ready:", npz_path, npz_path.exists())


## 2. The harness

One loader, one evaluator, one null. `evaluate(fit_predict, target)` takes any callable
`fit_predict(X_train, y_train, X_test) -> predictions` and scores it on the SAME
leave-one-subject-out folds every arm in the frozen table used. Both metric levels are
returned for every fold; nothing is silently mixed.


In [ ]:
import numpy as np
from scipy.stats import wilcoxon

HV = {"anger": ["SR_NA1", "SR_NA7"],
      "apprehension": ["SR_NA9", "SR_NA10"],
      "confusion": ["SR_PA2"]}

def _f(v):
    try:
        return float(v)
    except Exception:
        return None

def load(target):
    """Per-awakening raw epochs, binary label, subject id."""
    ensure_data()          # builds the dataset if the data cell was skipped
    z = zipfile.ZipFile(DATA_DIR / "REM_Turku.zip")
    rat = {r["Filename"]: r for r in csv.DictReader(
        io.StringIO(z.read("REM_Turku/Data/Ratings.csv").decode("utf-8-sig")))}
    rec = {r["Filename"]: r for r in csv.DictReader(
        io.StringIO(z.read("REM_Turku/Records.csv").decode("utf-8-sig")))}
    npz = np.load(DATA_DIR / "remturku_epochs.npz")
    X, y, s = [], [], []
    for fn, r in rat.items():
        k = f"{fn}|raw"
        if k not in npz or fn not in rec:
            continue
        if not any((_f(r[c]) or 0) > 0 for c in r if c.startswith("SR_")):
            continue
        X.append(np.asarray(npz[k], dtype=np.float32))
        y.append(int(any((_f(r[c]) or 0) > 0 for c in HV[target])))
        s.append(rec[fn]["Subject ID"])
    return X, np.array(y), np.array(s)

def bal(p, t):
    p, t = np.asarray(p), np.asarray(t)
    tp = ((p == 1) & (t == 1)).sum(); fn = ((p == 0) & (t == 1)).sum()
    tn = ((p == 0) & (t == 0)).sum(); fp = ((p == 1) & (t == 0)).sum()
    se = tp / (tp + fn) if tp + fn else 0.0
    sp = tn / (tn + fp) if tn + fp else 0.0
    return float((se + sp) / 2)

def evaluate(fit_predict, target, shuffle_seed=0, verbose=True):
    """LOSO. Returns {subject: {'epoch': bal, 'awk': bal}} over scorable subjects.
    shuffle_seed >= 1 permutes labels WITHIN subject (for nulls); 0 = observed."""
    Xf, y_awk, s_awk = load(target)
    y_awk = y_awk.copy()
    if shuffle_seed:
        assert shuffle_seed >= 1, "seed 0 means NO shuffle; null loops start at 1"
        rng = np.random.default_rng(shuffle_seed)
        for u in np.unique(s_awk):
            m = s_awk == u
            y_awk[m] = rng.permutation(y_awk[m])
    # epoch-level arrays with awakening index for aggregation
    Xe, ye, se_, ae = [], [], [], []
    for i, (x, yy, ss) in enumerate(zip(Xf, y_awk, s_awk)):
        Xe.append(x); ye += [int(yy)] * len(x); se_ += [ss] * len(x); ae += [i] * len(x)
    Xe = np.concatenate(Xe); ye = np.array(ye)
    se_ = np.array(se_); ae = np.array(ae)
    out = {}
    for held in np.unique(se_):
        tr, te = se_ != held, se_ == held
        if len(np.unique(ye[te])) < 2:
            continue            # unscorable: single test class
        pe = np.asarray(fit_predict(Xe[tr], ye[tr], Xe[te]))
        pa, ta = [], []
        for aw in np.unique(ae[te]):
            m = ae[te] == aw
            pa.append(int(pe[m].mean() > 0.5)); ta.append(int(ye[te][m][0]))
        out[str(held)] = {"epoch": bal(pe, ye[te]), "awk": bal(pa, ta)}
        if verbose:
            print(f"held={held}  epoch={out[str(held)]['epoch']:.3f}  "
                  f"awk={out[str(held)]['awk']:.3f}")
    return out

def paired(res_a, res_b, level="awk"):
    """Wilcoxon over the shared held-out subjects. n = subjects, never epochs."""
    common = sorted(set(res_a) & set(res_b), key=int)
    a = np.array([res_a[s][level] for s in common])
    b = np.array([res_b[s][level] for s in common])
    return {"n": len(common), "mean_a": a.mean(), "mean_b": b.mean(),
            "delta_pp": 100 * (a - b).mean(), "wins": int((a > b).sum()),
            "p": float(wilcoxon(a, b).pvalue) if len(common) >= 6 else float("nan")}


## 3. The arms

Each arm is a factory returning `fit_predict`. Add yours at the bottom; everything else
stays untouched.


In [ ]:
def make_tangent_lda(recentre=False):
    from pyriemann.estimation import Covariances
    from pyriemann.tangentspace import TangentSpace
    from pyriemann.utils.mean import mean_riemann
    from sklearn.discriminant_analysis import LinearDiscriminantAnalysis as LDA

    def _recentre(C):
        M = mean_riemann(C)
        w, V = np.linalg.eigh(M)
        Mi = V @ np.diag(1.0 / np.sqrt(np.maximum(w, 1e-12))) @ V.T
        return Mi @ C @ Mi

    def fit_predict(Xtr, ytr, Xte):
        cov = Covariances(estimator="oas")
        Ctr, Cte = cov.transform(Xtr), cov.transform(Xte)
        if recentre:
            Ctr, Cte = _recentre(Ctr), _recentre(Cte)   # label-free, legal under LOSO
        ts = TangentSpace().fit(Ctr)
        clf = LDA(solver="lsqr", shrinkage="auto").fit(ts.transform(Ctr), ytr)
        return clf.predict(ts.transform(Cte))
    return fit_predict


In [ ]:
sys.path.insert(0, "..")           # repo root: harness/ = the code that ran on the cluster
from harness.baseline_remturku import train_eval, grid_for, sample_configs

DL_DEFAULTS = dict(lr=1e-3, weight_decay=1e-4, epochs=30, batch_size=64,
                   scheduler="cycle", three_phase=True, pct_start=0.1,
                   patience=8, min_delta=1e-4,
                   F1=8, D=2, kernel_length=64, depthwise_kernel_length=64,
                   drop_prob=0.25, hidden_dim=256)

def make_dl(arch="shallow_bd", cfg=None, seed=0, val_frac=0.15):
    """Same optimizer / OneCycleLR three-phase scheduler / early stopping that produced
    the frozen table (harness/tuning_core.py, tuning_p10_v3 verbatim)."""
    cfg = dict(DL_DEFAULTS, **(cfg or {}))
    def fit_predict(Xtr, ytr, Xte):
        rng = np.random.default_rng(seed)
        idx = rng.permutation(len(Xtr)); nva = max(1, int(val_frac * len(Xtr)))
        va, tr = idx[:nva], idx[nva:]
        X4 = lambda X: X[..., None]     # harness expects (b, ch, t, 1)
        preds, _ = train_eval(X4(Xtr[tr]), ytr[tr], X4(Xtr[va]), ytr[va],
                              X4(Xte), np.zeros(len(Xte), dtype=int),
                              arch=arch, cfg=cfg, seed=seed)
        return np.asarray(preds)
    return fit_predict


In [ ]:
def make_tsmnet(epochs=40, lr=1e-3, batch=256, seed=0):
    """TSMNet (Kobler et al., NeurIPS 2022), authoritative repo, defaults.

    Device rule: the SPD stage runs on CPU by design, so bring the TARGET to the
    logits' device. Never force-move module tensors to cuda: the crash vanishes but the
    architecture silently changes."""
    import torch
    from spdnets.models import TSMNet
    from geoopt.optim import RiemannianAdam

    def fit_predict(Xtr, ytr, Xte):
        torch.manual_seed(seed)
        dev = "cuda" if torch.cuda.is_available() else "cpu"
        X = np.concatenate([Xtr, Xte])
        X = (X - X.mean(-1, keepdims=True)) / (X.std(-1, keepdims=True) + 1e-7)
        Xtr_n, Xte_n = X[:len(Xtr)], X[len(Xtr):]
        net = TSMNet(temporal_filters=4, spatial_filters=40, subspacedims=20,
                     nclasses=2, nchannels=Xtr.shape[1], nsamples=Xtr.shape[2],
                     domains=torch.arange(2)).to(dev)
        opt = RiemannianAdam(net.parameters(), lr=lr)
        lossf = torch.nn.CrossEntropyLoss()
        Xt = torch.from_numpy(Xtr_n).float(); yt = torch.from_numpy(ytr).long()
        d0 = torch.zeros(len(Xt)).long()
        idx = np.arange(len(Xt))
        net.train()
        for _ in range(epochs):
            np.random.shuffle(idx)
            for b in range(0, len(idx), batch):
                j = idx[b:b + batch]
                out = net(Xt[j].to(dev), d0[j].to(dev))
                logits = out[0] if isinstance(out, tuple) else out
                loss = lossf(logits, yt[j].to(logits.device))
                opt.zero_grad(); loss.backward(); opt.step()
        net.eval()
        with torch.no_grad():
            out = net(torch.from_numpy(Xte_n).float().to(dev),
                      torch.ones(len(Xte_n)).long().to(dev))
            logits = out[0] if isinstance(out, tuple) else out
            return logits.argmax(1).cpu().numpy()
    return fit_predict


## 4. Frozen results (2026-08-31), the table your method is scored against

Cross-subject LOSO, balanced accuracy, chance 0.50, n = scorable held-out subjects.
**Level** says which statistic the number is: `epoch` or `awakening` (the labelled unit;
the two can disagree, see trap 4).

| arm | level | apprehension | anger | confusion |
|---|---|---|---|---|
| TSMNet defaults (SPD) | awakening | **0.585 (14)**, null p=0.14 | 0.428 (16) | 0.513 (13) |
| TSMNet defaults (SPD) | epoch | 0.583 (14) | 0.435 (16) | 0.475 (13) |
| EEGNet defaults | epoch | 0.562 (14) | - | - |
| ShallowConv (bd) defaults | epoch | 0.546 (14) | 0.454 (16) | 0.507 (13) |
| ShallowConv nested-tuned | epoch | 0.557 (14) | - | 0.491 (13) |
| EEGNet nested-tuned | epoch | 0.530 (14) | 0.482 (16) | 0.542 (13) |
| tangent + LDA global | epoch | 0.552 (14) | 0.381 (16) | 0.482 (13) |
| tangent + LDA recentred | epoch | 0.512 (14) | 0.352 (16) | 0.437 (13) |

Significant results (each against its own null):

- **Tangent beats matched DL, paired: +2.9 pp, 11/14 subjects, Wilcoxon p = 0.012.**
- **TSMNet beats both tangent arms on anger, paired: p = 0.029 / 0.044.**
- **Anger decodes INVERTED: AUC 0.334 vs null 0.502, p = 0.010 (Bonferroni-passing)**,
  mechanism: between-subject r = +0.18, within-subject r = -0.24 (Simpson's paradox).
- Positive controls through the identical pipeline: subject identity 0.895 (17-way,
  null max 0.128), recording night 0.812 (null max 0.571). The nulls above are not a
  broken pipeline.

No arm clears chance on absolute dream-affect accuracy. **A method that does, under this
harness on these folds, is the headline of the paper.**


## 5. Reproduce the headline results

Three demos, cheapest first. Expected values from the frozen record are printed next to
each so you see immediately whether your run reproduces.

1. **The inverted anger ranking** (CPU, ~2 min): cross-subject AUC ~0.33 where chance is
   0.50: significantly BELOW chance (p = 0.010, Bonferroni-passing in the record). The
   decoder learns a between-subject direction that reverses within subject.
2. **Geometry beats deep learning, paired** (GPU, ~30 min): tangent vs ShallowConv on
   identical folds: expect delta ~ +3 pp, most subjects won (record: 11/14, p = 0.012).
3. **The best arm, TSMNet** (GPU, ~45 min): expect awakening-level mean ~0.585.


In [ ]:
# Demo 1: the inverted anger AUC (CPU, minutes). Record: 0.334, p=0.010.
from sklearn.metrics import roc_auc_score
from pyriemann.estimation import Covariances
from pyriemann.tangentspace import TangentSpace
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis as LDA

def tangent_auc(target):
    Xf, y_awk, s_awk = load(target)
    Xe, ye, se = [], [], []
    for x, yy, ss in zip(Xf, y_awk, s_awk):
        Xe.append(x); ye += [int(yy)] * len(x); se += [ss] * len(x)
    Xe = np.concatenate(Xe); ye = np.array(ye); se = np.array(se)
    aucs = []
    for held in np.unique(se):
        tr, te = se != held, se == held
        if len(np.unique(ye[te])) < 2:
            continue
        cov = Covariances(estimator="oas")
        Ctr, Cte = cov.transform(Xe[tr]), cov.transform(Xe[te])
        ts = TangentSpace().fit(Ctr)
        clf = LDA(solver="lsqr", shrinkage="auto").fit(ts.transform(Ctr), ye[tr])
        aucs.append(roc_auc_score(ye[te], clf.decision_function(ts.transform(Cte))))
        print(f"held={held}  AUC={aucs[-1]:.3f}")
    print(f"\nmean cross-subject AUC ({target}): {np.mean(aucs):.4f}  "
          f"(chance 0.50; frozen record 0.334, p=0.010 INVERTED)")
    return aucs

aucs_anger = tangent_auc("anger")


In [ ]:
# Demo 2 (GPU ~30 min): geometry vs deep learning, paired on identical folds.
# Record: tangent +2.9 pp over DL, 11/14 subjects, Wilcoxon p = 0.012.
res_tangent = evaluate(make_tangent_lda(), target="apprehension", verbose=False)
res_dl = evaluate(make_dl("shallow_bd"), target="apprehension", verbose=False)
print(paired(res_tangent, res_dl, level="epoch"))


In [ ]:
# Demo 3 (GPU ~45 min): the best arm. Record: TSMNet 0.585 awakening-level, n=14.
res_tsmnet = evaluate(make_tsmnet(), target="apprehension", verbose=True)
print("TSMNet awakening-level mean:",
      round(np.mean([v["awk"] for v in res_tsmnet.values()]), 4))


## 6. Your entry point

Write a factory, run both cells, and compare paired. That is the whole integration.

```python
def make_your_method(**kw):
    def fit_predict(Xtr, ytr, Xte):
        ...
        return predictions
    return fit_predict

res_yours = evaluate(make_your_method(), target="apprehension")
print(paired(res_yours, res_tangent, level="awk"))
```

Then the permutation null (`shuffle_seed = 1..20` to start), and only then a claim.
`PREREG.md` in the repo root holds every pre-registration and the deviation log;
`COMMIT_MAP_2026-08-31.txt` maps pre-rewrite hashes cited there to current ones.


## 7. body_action (101-Nights) data

Private single-subject data; never public. Two ways in, URL first:

- **URL (zero setup):** paste the two Google Drive share links below and run; the cell
  downloads both files directly (`gdown` handles large-file confirmation). Links come
  from Ninon.
- Fallback: mount your own Drive and point `BA_DIR` at wherever the two files live.


In [ ]:
BA_PT_URL = ""    # Drive share link for dream_dense_training_pairs_v3_256ch_fp16.pt
BA_CSV_URL = ""   # Drive share link for 101-Nights-Final-dreamsheet_LABELED_filter.csv
BA_DIR = Path("./body_action_data"); BA_DIR.mkdir(exist_ok=True)

if BA_PT_URL and BA_CSV_URL:
    subprocess.run([sys.executable, "-m", "pip", "-q", "install", "gdown"], check=True)
    import gdown
    gdown.download(BA_PT_URL, str(BA_DIR / "dream_dense_training_pairs_v3_256ch_fp16.pt"),
                   fuzzy=True, quiet=False)
    gdown.download(BA_CSV_URL, str(BA_DIR / "101-Nights-Final-dreamsheet_LABELED_filter.csv"),
                   fuzzy=True, quiet=False)
    md5 = hashlib.md5((BA_DIR / "dream_dense_training_pairs_v3_256ch_fp16.pt").read_bytes()).hexdigest()
    assert md5 == "c1b47af7da727515c4570ee1912861df", "tensor md5 mismatch: " + md5
    print("body_action data ready:", sorted(f.name for f in BA_DIR.iterdir()))
else:
    print("set BA_PT_URL and BA_CSV_URL (or mount Drive and point BA_DIR); skipping")
